# 04 - Dataset curado: musicas brasileiras populares

Reduz o subconjunto brasileiro do notebook 03 a um dataset menor, voltado a extracao posterior de letras e enriquecimento dos modelos de RI (BM25, TF-IDF, vetorial).

Camadas aplicadas, em ordem:
1. Filtro de genero brasileiro (mesmo regex do nb 03).
2. `track_popularity >= 30`.
3. `release_year >= 1990`.
4. Macrogenero derivado a partir de `artist_genres` (sertanejo, funk, pagode_samba, forro_arrocha, mpb_bossa_choro, axe_regional, rap_trap, phonk, rock_br, pop_br, gospel, outros).
5. Cap de 5 faixas por artista primario (min `artist_rowid` da faixa).
6. Top K por (macrogenero x decada).

ISRC com prefixo `BR` entra como **coluna auxiliar** (`isrc_br`), nao como filtro.

Saida: `data/derived/br_curated_tracks.parquet`.

## 0. Imports e parametros

In [1]:
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

DATASET_DIR = Path("../data/spotify-metadata")
CLEAN_DIR = DATASET_DIR / "spotify_clean_parquet"
OUTPUT_DIR = Path("../data/derived")
OUTPUT_PATH = OUTPUT_DIR / "br_curated_tracks.parquet"

POPULARITY_MIN = 30
RELEASE_YEAR_MIN = 1990
PER_ARTIST_CAP = 5
TOP_K_PER_BUCKET = 250

BR_TERMS = [
    "brazilian", "brazil", "brasil", "mpb", "samba", "sertanejo", "pagode",
    "forro", "bossa", "axe", "baiao", "funk carioca", "arrocha", "piseiro",
    "frevo", "maracatu", "tropicalia", "choro", "musica brasileira",
]
BR_REGEX = r"(^|[^a-z])(" + "|".join(BR_TERMS) + r")([^a-z]|$)"


def sql_path(path: Path) -> str:
    return path.as_posix().replace("'", "''")


required_files = [
    CLEAN_DIR / "tracks.parquet",
    CLEAN_DIR / "track_artists.parquet",
    CLEAN_DIR / "artists.parquet",
    CLEAN_DIR / "artist_genres.parquet",
    CLEAN_DIR / "albums.parquet",
]
missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Arquivos ausentes em data/spotify-metadata. "
        "Rode `./scripts/download_spotify_metadata.sh --truncated` ou `--full`.\n"
        + "\n".join(str(path) for path in missing)
    )

tracks_path = sql_path(CLEAN_DIR / "tracks.parquet")
track_artists_path = sql_path(CLEAN_DIR / "track_artists.parquet")
artists_path = sql_path(CLEAN_DIR / "artists.parquet")
artist_genres_path = sql_path(CLEAN_DIR / "artist_genres.parquet")
albums_path = sql_path(CLEAN_DIR / "albums.parquet")

con = duckdb.connect()
print(
    f"popularity_min={POPULARITY_MIN} | year_min={RELEASE_YEAR_MIN} | "
    f"per_artist_cap={PER_ARTIST_CAP} | top_k_per_bucket={TOP_K_PER_BUCKET}"
)
print(f"BR_REGEX: {BR_REGEX}")

popularity_min=30 | year_min=1990 | per_artist_cap=5 | top_k_per_bucket=250
BR_REGEX: (^|[^a-z])(brazilian|brazil|brasil|mpb|samba|sertanejo|pagode|forro|bossa|axe|baiao|funk carioca|arrocha|piseiro|frevo|maracatu|tropicalia|choro|musica brasileira)([^a-z]|$)


## 1. Camada 1: filtro de genero brasileiro

Mesmo criterio do notebook 03 — qualquer artista da faixa com genero que case o regex `BR_REGEX` apos `lower(strip_accents(...))`.

In [2]:
con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE br_track_ids AS
    SELECT DISTINCT ta.track_rowid
    FROM read_parquet('{track_artists_path}') ta
    JOIN read_parquet('{artist_genres_path}') ag
      ON ag.artist_rowid = ta.artist_rowid
    WHERE regexp_matches(lower(strip_accents(coalesce(ag.genre, ''))), '{BR_REGEX}')
    """
)
n_camada1 = con.execute("SELECT count(*) FROM br_track_ids").fetchone()[0]
print(f"Camada 1 (genero BR): {n_camada1:,} faixas")

Camada 1 (genero BR): 3,418,482 faixas


## 2. Camadas 2 e 3: popularidade >= 30 e ano >= 1990

Materializa `br_candidates` com:
- artista primario = `min(artist_rowid)` da faixa (determinismo, sem coluna de posicao);
- todos os nomes de artistas concatenados em `artist_names`;
- todos os generos dos artistas concatenados em `artist_genres`;
- coluna `isrc_br` derivada do prefixo dos 2 primeiros caracteres do ISRC.

In [3]:
con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE br_candidates AS
    WITH artist_profile AS (
        SELECT
            ta.track_rowid,
            min(ta.artist_rowid) AS primary_artist_rowid,
            string_agg(DISTINCT ar.name, ' | ' ORDER BY ar.name) AS artist_names
        FROM br_track_ids b
        JOIN read_parquet('{track_artists_path}') ta
          ON ta.track_rowid = b.track_rowid
        LEFT JOIN read_parquet('{artists_path}') ar
          ON ar.rowid = ta.artist_rowid
        GROUP BY ta.track_rowid
    ),
    genre_profile AS (
        SELECT
            ta.track_rowid,
            string_agg(DISTINCT ag.genre, ' | ' ORDER BY ag.genre) AS artist_genres
        FROM br_track_ids b
        JOIN read_parquet('{track_artists_path}') ta
          ON ta.track_rowid = b.track_rowid
        JOIN read_parquet('{artist_genres_path}') ag
          ON ag.artist_rowid = ta.artist_rowid
        WHERE coalesce(ag.genre, '') <> ''
        GROUP BY ta.track_rowid
    )
    SELECT
        t.id AS track_id,
        t.rowid AS track_rowid,
        coalesce(t.name, '') AS track_name,
        t.external_id_isrc AS isrc,
        CASE
            WHEN substr(upper(coalesce(t.external_id_isrc, '')), 1, 2) = 'BR'
            THEN TRUE ELSE FALSE
        END AS isrc_br,
        ap.primary_artist_rowid,
        par.name AS primary_artist_name,
        coalesce(ap.artist_names, '') AS artist_names,
        coalesce(gp.artist_genres, '') AS artist_genres,
        alb.id AS album_id,
        coalesce(alb.name, '') AS album_name,
        coalesce(alb.release_date, '') AS release_date,
        try_cast(regexp_extract(coalesce(alb.release_date, ''), '^[0-9]{{4}}') AS INTEGER) AS release_year,
        t.popularity AS track_popularity,
        t.duration_ms,
        cast(t.explicit AS BOOLEAN) AS explicit
    FROM br_track_ids b
    JOIN read_parquet('{tracks_path}') t
      ON t.rowid = b.track_rowid
    LEFT JOIN read_parquet('{albums_path}') alb
      ON alb.rowid = t.album_rowid
    LEFT JOIN artist_profile ap
      ON ap.track_rowid = t.rowid
    LEFT JOIN read_parquet('{artists_path}') par
      ON par.rowid = ap.primary_artist_rowid
    LEFT JOIN genre_profile gp
      ON gp.track_rowid = t.rowid
    WHERE t.popularity >= {POPULARITY_MIN}
      AND try_cast(regexp_extract(coalesce(alb.release_date, ''), '^[0-9]{{4}}') AS INTEGER) >= {RELEASE_YEAR_MIN}
    """
)
n_camada23 = con.execute("SELECT count(*) FROM br_candidates").fetchone()[0]
print(f"Camadas 2+3 (pop >= {POPULARITY_MIN} e ano >= {RELEASE_YEAR_MIN}): {n_camada23:,} faixas")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Camadas 2+3 (pop >= 30 e ano >= 1990): 102,614 faixas


## 3. Camada 4: macrogenero derivado

Reduz centenas de microgeneros a 12 macrobalas por priorizacao. Ordem importa: `funk` vem antes de `axe_regional` para que `brega funk` caia em funk; `gospel` vem primeiro para nao ser engolido por outras categorias.

Tambem cria a coluna `decade` (`(release_year / 10) * 10`).

In [4]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE br_with_macro AS
    SELECT
        *,
        (release_year / 10) * 10 AS decade,
        CASE
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(gospel)([^a-z]|$)') THEN 'gospel'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(sertanejo|agronejo)([^a-z]|$)') THEN 'sertanejo'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(funk)([^a-z]|$)') THEN 'funk'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(forro|arrocha|piseiro|seresta|baiao)([^a-z]|$)') THEN 'forro_arrocha'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(pagode|samba)([^a-z]|$)') THEN 'pagode_samba'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(mpb|bossa|choro|tropicalia)([^a-z]|$)') THEN 'mpb_bossa_choro'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(axe|frevo|maracatu|brega|tecnobrega)([^a-z]|$)') THEN 'axe_regional'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(rap|trap|hip hop)([^a-z]|$)') THEN 'rap_trap'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(phonk)([^a-z]|$)') THEN 'phonk'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(rock)([^a-z]|$)') THEN 'rock_br'
            WHEN regexp_matches(lower(strip_accents(artist_genres)), '(^|[^a-z])(pop)([^a-z]|$)') THEN 'pop_br'
            ELSE 'outros'
        END AS macro_genre
    FROM br_candidates
    """
)
macro_dist = con.execute(
    """
    SELECT macro_genre, count(*) AS faixas
    FROM br_with_macro
    GROUP BY macro_genre
    ORDER BY faixas DESC
    """
).fetchdf()
macro_dist

,macro_genre,faixas
0,sertanejo,19675
1,funk,17489
2,mpb_bossa_choro,15918
3,pagode_samba,10813
4,gospel,10061
5,forro_arrocha,9424
6,phonk,7489
7,rap_trap,6633
8,outros,2286
9,rock_br,1179


## 4. Camadas 5 e 6: cap por artista + top K por (macrogenero x decada)

Primeiro cap de `PER_ARTIST_CAP` por artista primario (mantem as mais populares de cada um), depois top `TOP_K_PER_BUCKET` por celula `macro_genre x decade`. Empates resolvidos por `track_rowid` para determinismo.

In [5]:
con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE br_after_artist_cap AS
    SELECT *
    FROM (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY primary_artist_rowid
                ORDER BY track_popularity DESC, track_rowid
            ) AS rn_artist
        FROM br_with_macro
    )
    WHERE rn_artist <= {PER_ARTIST_CAP}
    """
)
n_apos_cap = con.execute("SELECT count(*) FROM br_after_artist_cap").fetchone()[0]
print(f"Apos cap de {PER_ARTIST_CAP} por artista: {n_apos_cap:,} faixas")

con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE br_curated AS
    SELECT *
    FROM (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY macro_genre, decade
                ORDER BY track_popularity DESC, track_rowid
            ) AS rn_bucket
        FROM br_after_artist_cap
    )
    WHERE rn_bucket <= {TOP_K_PER_BUCKET}
    """
)
n_final = con.execute("SELECT count(*) FROM br_curated").fetchone()[0]
print(f"Apos top {TOP_K_PER_BUCKET} por (macrogenero x decada): {n_final:,} faixas")

Apos cap de 5 por artista: 33,329 faixas
Apos top 250 por (macrogenero x decada): 22,220 faixas


## 5. Sanity checks

In [6]:
from IPython.display import display

print("Distribuicao por (macrogenero x decada):")
display(
    con.execute(
        """
        SELECT macro_genre, decade, count(*) AS faixas, avg(track_popularity)::INTEGER AS pop_media
        FROM br_curated
        GROUP BY macro_genre, decade
        ORDER BY macro_genre, decade
        """
    ).fetchdf()
)

print("\nResumo por macrogenero:")
display(
    con.execute(
        """
        SELECT
            macro_genre,
            count(*) AS faixas,
            count(DISTINCT primary_artist_rowid) AS artistas_distintos,
            avg(track_popularity)::INTEGER AS pop_media,
            min(release_year) AS ano_min,
            max(release_year) AS ano_max,
            100.0 * sum(CASE WHEN isrc_br THEN 1 ELSE 0 END) / count(*) AS pct_isrc_br
        FROM br_curated
        GROUP BY macro_genre
        ORDER BY faixas DESC
        """
    ).fetchdf().round(1)
)

print("\nTop 15 artistas no dataset curado:")
display(
    con.execute(
        """
        SELECT primary_artist_name AS artista, count(*) AS faixas, avg(track_popularity)::INTEGER AS pop_media
        FROM br_curated
        WHERE primary_artist_name IS NOT NULL
        GROUP BY primary_artist_name
        ORDER BY faixas DESC, pop_media DESC
        LIMIT 15
        """
    ).fetchdf()
)

print("\nTop 15 faixas mais populares no dataset curado:")
display(
    con.execute(
        """
        SELECT track_name, primary_artist_name, macro_genre, release_year, track_popularity, isrc_br
        FROM br_curated
        ORDER BY track_popularity DESC, track_rowid
        LIMIT 15
        """
    ).fetchdf()
)

Distribuicao por (macrogenero x decada):


,macro_genre,decade,faixas,pop_media
0,axe_regional,1990.0,2,41
1,axe_regional,1996.0,5,44
2,axe_regional,1997.0,4,49
3,axe_regional,1998.0,1,35
4,axe_regional,1999.0,14,38
...,...,...,...,...
370,sertanejo,2021.0,129,45
371,sertanejo,2022.0,158,44
372,sertanejo,2023.0,250,45
373,sertanejo,2024.0,250,63



Resumo por macrogenero:


,macro_genre,faixas,artistas_distintos,pop_media,ano_min,ano_max,pct_isrc_br
0,mpb_bossa_choro,4171,1629,43,1990,2025,35.2
1,gospel,2914,1034,42,1990,2025,51.3
2,pagode_samba,2436,929,42,1990,2025,54.6
3,funk,2311,1166,52,1993,2025,16.1
4,forro_arrocha,2229,896,41,1990,2025,31.4
5,sertanejo,2191,854,48,1990,2025,47.3
6,rap_trap,2133,998,42,1994,2025,16.6
7,outros,1611,853,39,1990,2025,9.9
8,phonk,877,406,53,2015,2025,0.1
9,axe_regional,551,226,38,1990,2025,41.7



Top 15 artistas no dataset curado:


,artista,faixas,pop_media
0,Leonardo,12,46
1,Zezo,10,52
2,Banda Magníficos,10,49
3,Ventania,10,44
4,Facção Central,10,43
5,Robério e Seus Teclados,10,42
6,Derek,8,48
7,Ursão,8,45
8,Matheusinho,8,38
9,RD12,7,58



Top 15 faixas mais populares no dataset curado:


,track_name,primary_artist_name,macro_genre,release_year,track_popularity,isrc_br
0,São Paulo (feat. Anitta),Anitta,funk,2025,87,False
1,Mãe Solteira,DG e Batidão Stronda,funk,2025,85,False
2,Resenha do Arrocha,J. Eskine,forro_arrocha,2024,85,False
3,Oh Garota Eu Quero Você Só Pra Mim,Zé Felipe,sertanejo,2024,85,False
4,Fui Mlk,FamousKyo,funk,2024,84,False
5,Tubarões - Ao Vivo,Diego & Victor Hugo,sertanejo,2025,84,False
6,FUNK DO BOUNCE (Slowed),Ariis,phonk,2024,83,False
7,Funk de Beleza - Slowed,Scythermane,phonk,2024,83,False
8,Descer,Dj LK da Escócia,funk,2025,83,False
9,Sei Que Tu Me Odeia,Anitta,sertanejo,2024,83,True


## 6. Exportar parquet

Salva apenas as colunas relevantes para a etapa seguinte (extracao de letras + indexacao). Schema final inclui o ISRC para servir como chave alternativa de matching com APIs externas.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_sql_path = sql_path(OUTPUT_PATH)

con.execute(
    f"""
    COPY (
        SELECT
            track_id,
            isrc,
            isrc_br,
            track_name,
            primary_artist_name,
            artist_names,
            artist_genres,
            macro_genre,
            album_id,
            album_name,
            release_date,
            release_year,
            decade,
            track_popularity,
            duration_ms,
            explicit
        FROM br_curated
        ORDER BY track_popularity DESC, track_rowid
    ) TO '{output_sql_path}' (FORMAT PARQUET)
    """
)

size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"Exportado: {OUTPUT_PATH} ({size_mb:.2f} MB, {n_final:,} faixas)")

Exportado: ..\data\derived\br_curated_tracks.parquet (2.76 MB, 22,220 faixas)
